# Deep Q-Network (DQN) on FrozenLake

## 1. Introduction

Tabular Q-Learning stores action values in a table, which becomes impractical when the state space is large or continuous.

**Deep Q-Network (DQN)** addresses this limitation by using a neural network to approximate the action-value function.

Instead of storing a separate value for every state-action pair, the network takes a state as input and predicts Q-values for all available actions.

For FrozenLake, we represent each of the 16 states using a one-hot vector and train a neural network to predict the four action values.

The objective is to learn a policy that maximizes expected discounted reward.

## 3. Environment Setup

We use Gymnasium's 4 × 4 FrozenLake environment. Each state is represented by an integer from 0 to 15, and the agent can choose one of four actions.

A neural network expects numerical feature vectors as input, so we'll convert each state into a one-hot encoded vector.

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False
)

n_states = env.observation_space.n
n_actions = env.action_space.n

print("States:", n_states)
print("Actions:", n_actions)
print("PyTorch:", torch.__version__)

States: 16
Actions: 4
PyTorch: 2.14.0+cu130


## 4. Device Selection

We use CUDA when available and otherwise fall back to the CPU. For this small environment, either device is sufficient; GPU acceleration is not guaranteed to be faster.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 5. One-Hot State Representation

Each state is represented by a vector of length 16, containing one `1` at the state's index and `0` everywhere else.

For state $s$, the encoded vector is:

$$
x_i =
\begin{cases}
1, & i=s,\\
0, & i\ne s.
\end{cases}
$$

For example, state 3 becomes $[0,0,0,1,0,\ldots,0]$.

This representation lets the neural network process discrete states as numerical inputs.

In [3]:
def encode_state(state):
    vector = np.zeros(n_states, dtype=np.float32)
    vector[state] = 1.0
    return torch.tensor(vector, device=device)

# Test the encoding
state, info = env.reset()
encoded = encode_state(state)

print("State:", state)
print("Encoded vector:", encoded)
print("Vector shape:", encoded.shape)

State: 0
Encoded vector: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
Vector shape: torch.Size([16])


## 6. Building the Q-Network

The network approximates the optimal action-value function using parameters $\theta$:

$$
Q(s,a;\theta)\approx Q^*(s,a)
$$

**Architecture:**

- **Input layer:** 16 neurons (one-hot encoded state).
- **Hidden layer:** 64 neurons with ReLU activation.
- **Output layer:** 4 neurons, one for each action.

For a state $s$, the network predicts:

$$
\mathbf{Q}(s;\theta)=
[Q(s,0;\theta),Q(s,1;\theta),Q(s,2;\theta),Q(s,3;\theta)]
$$

The selected greedy action is:

$$
a^*=\arg\max_a Q(s,a;\theta)
$$

We do not apply softmax to the output because Q-values represent expected returns, not probabilities.

In [6]:
class QNetwork(nn.Module):
    def __init__(self, n_states, n_actions):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_states, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )

    def forward(self, x):
        return self.network(x)

## 8. DQN: Understanding the Target and Loss Function

### 1. What are we trying to learn?

Our neural network takes a state $s$ as input and predicts four Q-values, one for each action.

Suppose the agent is in state $s=5$ and chooses action $a=2$ (Right). The network predicts:

$$
Q(s,a;\theta)=0.30
$$

This is the network's current estimate of how valuable that action is.

But how do we know whether $0.30$ is a good estimate? We use the reward and the next state's predicted value to construct a **target**.

### 2. Deriving the target

From the Q-Learning update, the target is:

$$
Y = r+\gamma\max_{a'}Q(s',a')
$$

Here:

- $r$: reward received after taking the action.
- $s'$: the next state.
- $\gamma$: discount factor, controlling the importance of future rewards.
- $\max_{a'}Q(s',a')$: the highest estimated Q-value available in the next state.

The target tells us what the current Q-value should move toward, based on the observed transition.

**Example:** Suppose the agent receives reward $r=0$, the discount factor is $\gamma=0.9$, and the best next-state Q-value is $0.8$.

Then:

$$
Y=0+0.9(0.8)=0.72
$$

Our network predicted $0.30$, but the target is $0.72$. We want the prediction to move closer to $0.72$.

### 3. Why do we need a loss function?

The loss measures how far the network's prediction is from its target.

$$
L(\theta)=\left[Y-Q(s,a;\theta)\right]^2
$$

Using our example:

$$
L=(0.72-0.30)^2=0.1764
$$

A large error gives a larger loss. During training, backpropagation and the optimizer adjust the network parameters $\theta$ to reduce this loss.

The target is not necessarily the perfect answer; it is a learning target calculated from the current estimates and observed reward.

### 4. Why does DQN use a target network?

If we calculate the target using the same network that we are updating, both the prediction and target can change together during training. This can make learning unstable.

DQN therefore uses two networks:

- **Online network**, with parameters $\theta$: predicts the current Q-value and gets trained.
- **Target network**, with parameters $\theta^-$: estimates the next state's Q-values to calculate the target.

The target becomes:

$$
Y =
r+\gamma(1-d)\max_{a'}Q(s',a';\theta^-)
$$

Here, $d=1$ if the transition ends the episode and $d=0$ otherwise.

The factor $(1-d)$ removes the future-value term for terminal transitions. The target network's parameters are copied from the online network periodically rather than after every update.

**In summary:** The online network predicts a value, the target provides a learning reference, and the loss measures their difference. Training reduces that difference over time.

In [7]:
q_network = QNetwork(n_states, n_actions).to(device)

print(q_network)

QNetwork(
  (network): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=4, bias=True)
  )
)


In [8]:
import copy

# Target network starts with the same weights as the online network
target_network = copy.deepcopy(q_network)
target_network.eval()

# Optimizer updates only the online network
optimizer = optim.Adam(q_network.parameters(), lr=0.001)

# Measures the difference between predicted Q-value and target
loss_fn = nn.MSELoss()

print("Online network and target network initialized.")
print("Device:", device)

Online network and target network initialized.
Device: cuda


## 9. Experience Replay

During training, the agent collects transitions of the form:

$$
(s_t,a_t,r_{t+1},s_{t+1},d_t)
$$

Instead of learning only from the latest transition, DQN stores transitions in a replay buffer and samples random mini-batches for training.

**Why?**

- **Breaks temporal correlation:** Consecutive transitions are often similar; random sampling makes training batches less correlated.
- **Reuses experience:** A transition can contribute to multiple training updates.
- **Stabilizes learning:** Mini-batches provide a more varied learning signal.

The replay buffer has a maximum capacity. When it fills up, older transitions are discarded as new ones arrive.

In [9]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append(
            (state, action, reward, next_state, done)
        )

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32)
        )

    def __len__(self):
        return len(self.buffer)

In [10]:
replay_buffer = ReplayBuffer(capacity=10_000)
batch_size = 64

print("Replay buffer initialized.")
print("Capacity:", replay_buffer.buffer.maxlen)

Replay buffer initialized.
Capacity: 10000


## 10. The DQN Learning Step

We sample a mini-batch of transitions from the replay buffer:

$$
(s,a,r,s',d)
$$

For each transition, the target network estimates the next state's Q-values. The target is:

$$
y=r+\gamma(1-d)\max_{a'}Q(s',a';\theta^-)
$$

The online network predicts $Q(s,a;\theta)$ for the action actually taken. We minimize the mean squared error between predictions and targets:

$$
L(\theta)=\frac{1}{B}\sum_{i=1}^{B}
\left[y_i-Q(s_i,a_i;\theta)\right]^2
$$

Here, $B$ is the batch size. Backpropagation updates the online network, while the target network remains unchanged during this step.

In [11]:
def learn():
    if len(replay_buffer) < batch_size:
        return None

    states, actions, rewards, next_states, dones = (
        replay_buffer.sample(batch_size)
    )

    states = torch.tensor(states, dtype=torch.float32, device=device)
    actions = torch.tensor(actions, dtype=torch.long, device=device)
    rewards = torch.tensor(rewards, dtype=torch.float32, device=device)
    next_states = torch.tensor(next_states, dtype=torch.float32, device=device)
    dones = torch.tensor(dones, dtype=torch.float32, device=device)

    # Q-values for the actions actually taken
    current_q = q_network(states).gather(
        1, actions.unsqueeze(1)
    ).squeeze(1)

    # Target values; no gradients through the target calculation
    with torch.no_grad():
        max_next_q = target_network(next_states).max(dim=1).values
        targets = rewards + gamma * (1 - dones) * max_next_q

    loss = loss_fn(current_q, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [12]:
def state_vector(state):
    vector = np.zeros(n_states, dtype=np.float32)
    vector[state] = 1.0
    return vector

### Updating the Target Network

The target network is copied from the online network periodically rather than after every gradient update.

$$
\theta^- \leftarrow \theta
$$

This helps keep the learning targets more stable while the online network changes.

In [13]:
def update_target_network():
    target_network.load_state_dict(q_network.state_dict())

## 11. Training the DQN Agent

In each episode, the agent interacts with FrozenLake and stores every transition in the replay buffer. Once enough transitions are available, it samples mini-batches and updates the online network.

The target network is synchronized periodically, and the exploration rate $\varepsilon$ gradually decreases.

We track episode rewards, success rates, and training losses to monitor learning.

In [16]:
import random
from collections import deque

# Reproducibility
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

# Training hyperparameters
num_episodes = 10_000
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.9995
target_update_frequency = 250

episode_rewards = []
episode_successes = []
training_losses = []

for episode in range(num_episodes):
    state, info = env.reset()
    total_reward = 0

    while True:
        # Convert state to one-hot vector
        state_vec = state_vector(state)

        # Epsilon-greedy action selection
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                state_tensor = torch.tensor(
                    state_vec, dtype=torch.float32, device=device
                )
                action = q_network(state_tensor).argmax().item()

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        next_state_vec = state_vector(next_state)

        # Store the transition
        replay_buffer.push(
            state_vec, action, reward, next_state_vec, float(done)
        )

        # Learn from a random mini-batch
        loss = learn()
        if loss is not None:
            training_losses.append(loss)

        state = next_state
        total_reward += reward

        if done:
            break

    episode_rewards.append(total_reward)
    episode_successes.append(int(total_reward > 0))

    # Periodically synchronize target network
    if (episode + 1) % target_update_frequency == 0:
        update_target_network()

    # Reduce exploration
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if (episode + 1) % 1000 == 0:
        recent_success = np.mean(episode_successes[-500:])
        print(
            f"Episode {episode + 1:5d} | "
            f"Success rate (last 500): {recent_success:.3f} | "
            f"Epsilon: {epsilon:.3f}"
        )

print("Training complete!")

Episode  1000 | Success rate (last 500): 0.088 | Epsilon: 0.606
Episode  2000 | Success rate (last 500): 0.502 | Epsilon: 0.368
Episode  3000 | Success rate (last 500): 0.730 | Epsilon: 0.223
Episode  4000 | Success rate (last 500): 0.822 | Epsilon: 0.135
Episode  5000 | Success rate (last 500): 0.908 | Epsilon: 0.082
Episode  6000 | Success rate (last 500): 0.950 | Epsilon: 0.050
Episode  7000 | Success rate (last 500): 0.982 | Epsilon: 0.030
Episode  8000 | Success rate (last 500): 0.984 | Epsilon: 0.018
Episode  9000 | Success rate (last 500): 0.978 | Epsilon: 0.011
Episode 10000 | Success rate (last 500): 0.990 | Epsilon: 0.010
Training complete!


## 12. Evaluating the Learned Policy

During training, the agent uses an $\varepsilon$-greedy policy, which includes random exploratory actions.

For evaluation, we disable exploration and select the action with the highest predicted Q-value:

$$
a_t=\arg\max_a Q(s_t,a;\theta)
$$

We run multiple independent episodes and calculate the success rate to estimate how reliably the learned policy reaches the goal.

In [17]:
def evaluate_dqn(env, q_network, num_episodes=1000):
    successes = 0
    episode_lengths = []

    for _ in range(num_episodes):
        state, info = env.reset()
        steps = 0

        while True:
            state_vec = state_vector(state)
            state_tensor = torch.tensor(
                state_vec, dtype=torch.float32, device=device
            )

            with torch.no_grad():
                action = q_network(state_tensor).argmax().item()

            state, reward, terminated, truncated, info = env.step(action)
            steps += 1

            if terminated or truncated:
                successes += int(reward > 0)
                episode_lengths.append(steps)
                break

    return successes / num_episodes, np.mean(episode_lengths)


success_rate, avg_steps = evaluate_dqn(env, q_network)

print(f"Evaluation success rate: {success_rate:.3f}")
print(f"Average episode length: {avg_steps:.2f} steps")

Evaluation success rate: 1.000
Average episode length: 6.00 steps


In [19]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True
)

slippery_rate, slippery_steps = evaluate_dqn(
    slippery_env, q_network
)

print(f"Slippery success rate: {slippery_rate:.3f}")
print(f"Average episode length: {slippery_steps:.2f} steps")

Slippery success rate: 0.036
Average episode length: 4.81 steps
